### 1. Setup de Ambiente
Criação do **catalog**, schemas **landing** e **bronze** e volume **landing.landing_volume**, cópia dos arquivos .csv do input para o volume **landing_volume**.

In [0]:
catalog = "cinedata"
landing_schema = "landing"
bronze_schema = "bronze"
landing_volume = "landing_volume"

In [0]:
create_catalog = f"CREATE CATALOG IF NOT EXISTS {catalog}"
create_landing_schema = f"CREATE SCHEMA IF NOT EXISTS {catalog}.{landing_schema}"
create_bronze_schema = f"CREATE SCHEMA IF NOT EXISTS {catalog}.{bronze_schema}"
create_landing_volume = f"CREATE VOLUME IF NOT EXISTS {catalog}.{landing_schema}.{landing_volume}"

In [0]:
spark.sql(create_catalog)
spark.sql(create_landing_schema)
spark.sql(create_bronze_schema)
spark.sql(create_landing_volume)

In [0]:
# Caminho dos arquivos de entrada
input_path = f"/Workspace/Users/murilomega100@gmail.com/CineData/input_files"

volume_path = f"/Volumes/{catalog}/{landing_schema}/{landing_volume}"

In [0]:
try:
    for file in dbutils.fs.ls(input_path):
        f_path = f"{volume_path}/{file.name}"
        dbutils.fs.cp(file.path, f_path)
        print(f"Arquivo {file.name} foi copiado para {volume_path}.")
    print("Arquivos copiados com sucesso.")
except Exception as e:
    print(f"Ocorreu um erro ao copiar os arquivos: {e}")

### 2. Ingestão dos dados: Landing -> Bronze

In [0]:
# Leitura dos arquivos .csv separadas em dataframes
df_movies_info = spark.read.csv(
    path=f"{volume_path}/movies_info_TMDB_IMDB.csv",
    header=True,
    inferSchema=True
)

df_movies_financials = spark.read.csv(
    path=f"{volume_path}/movies_financials_IMDB_TMDB.csv",
    header=True,
    inferSchema=True
)

df_movies_metrics = spark.read.csv(
    path=f"{volume_path}/movies_metrics_IMDB_TMDB.csv",
    header=True,
    inferSchema=True
)

df_credits_and_tags = spark.read.csv(
    path=f"{volume_path}/credits_and_tags_IMDB_TMDB.csv",
    header=True,
    inferSchema=True
)

df_movies_reviews = spark.read.csv(
    path=f"{volume_path}/movies_reviews.csv",
    header=True,
    inferSchema=True
)

In [0]:
# Adição da coluna 'ingestion_datetime' nos dataframes
from pyspark.sql.functions import current_timestamp

df_movies_info = df_movies_info.withColumn("ingestion_datetime", current_timestamp())
df_movies_financials = df_movies_financials.withColumn("ingestion_datetime", current_timestamp())
df_movies_metrics = df_movies_metrics.withColumn("ingestion_datetime", current_timestamp())
df_credits_and_tags = df_credits_and_tags.withColumn("ingestion_datetime", current_timestamp())
df_movies_reviews = df_movies_reviews.withColumn("ingestion_datetime", current_timestamp())

In [0]:
# Salvar como tabelas Delta
df_movies_info.write.format("delta").mode("append").saveAsTable(f"{catalog}.{bronze_schema}.tb_movies_info")
df_movies_financials.write.format("delta").mode("append").saveAsTable(f"{catalog}.{bronze_schema}.tb_movies_financials")
df_movies_metrics.write.format("delta").mode("append").saveAsTable(f"{catalog}.{bronze_schema}.tb_movies_metrics")
df_credits_and_tags.write.format("delta").mode("append").saveAsTable(f"{catalog}.{bronze_schema}.tb_credits_and_tags")
df_movies_reviews.write.format("delta").mode("append").saveAsTable(f"{catalog}.{bronze_schema}.tb_movies_reviews")

### 3. Ingestão de API

In [0]:
import requests

def get_cotacao_dolar(url:str) -> list:
    res = requests.get(url)
    if res.status_code != 200:
        print("Erro na requisição: " + res.status_code)
        return []
    data = res.json()
    return data["value"]

In [0]:
url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{dbutils.widgets.get("data_inicio_formatada")}'&@dataFinalCotacao='{dbutils.widgets.get("data_fim_formatada")}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

df_cotacao_dolar = spark.createDataFrame(get_cotacao_dolar(url))
df_cotacao_dolar.write.format("delta").mode("append").saveAsTable(f"{catalog}.{bronze_schema}.tb_cotacao_dolar")